In [11]:
# !pip install transformers
!pip install torch
!pip install scikit-learn

In [7]:
import unicodedata 
from typing import List, Tuple
from difflib import SequenceMatcher

In [9]:
def evaluate_encoding_accuracy(original: str, encoded: bytes, encoding: str = "utf-8") -> dict:
  decoded = encoded.decode(encoding, errors="replace")

  exact_match = original == decoded
  similarity = SequenceMatcher(None, original, decoded).ratio()

  byte_length = len(encoded)
  return {
          "exact_match": exact_match,
          "similarity_score": round(similarity, 4),
          "original_length": len(original),
          "byte_length": byte_length,
          "emoji_count": sum(1 for c in original if unicodedata.category(c).startswith('So'))
  }

In [10]:
def encode_text(text: str, encoding: str = "utf-8") -> bytes:
  try:
    encoded = text.encode(encoding, errors='replace')
    print(f"Encode successfully ({encoding}): {len(encoded)} bytes")
    return encoded
  except UnicodeEncodeError as e:
    print(f"Encode error: {e}. Try again with UTF-8")
    return text.encode("utf-8", errors="replace")

def decode_text(encoded_bytes: bytes, encoding: str = "utf-8") -> str:
  return encoded_bytes.decode(encoding, errors="replace")

text = "Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻"
print("Câu gốc:", text)

bytes_data = encode_text(text)

decoded = decode_text(bytes_data)
print("Decode lại:", decoded)
print("Checking the similarity between encode and decode: ", text == decoded)  
print(evaluate_encoding_accuracy(text, bytes_data ))

emoji_complex = "👨‍👩‍👧‍👦🏽"  
print("Complex emoji:", emoji_complex)
print("Code points:", [hex(ord(c)) for c in emoji_complex])

Câu gốc: Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻
Encode successfully (utf-8): 75 bytes
Decode lại: Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻
Checking the similarity between encode and decode:  True
{'exact_match': True, 'similarity_score': 1.0, 'original_length': 50, 'byte_length': 75, 'emoji_count': 5}
Complex emoji: 👨‍👩‍👧‍👦🏽
Code points: ['0x1f468', '0x200d', '0x1f469', '0x200d', '0x1f467', '0x200d', '0x1f466', '0x1f3fd']


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
tokenizer.add_tokens(["😊", "🌍", "❤️"])  

tokens = tokenizer.encode("Hello 😊", add_special_tokens=True)
print("Tokens:", tokens)
print("Decode again:", tokenizer.decode(tokens))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokens: [101, 31178, 119547, 102]
Decode lại: [CLS] Hello 😊 [SEP]


# How emoji attack the complexity of machine

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import tiktoken
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def analyze_tokenization(text_no_emoji: str, text_with_emoji: str):
  print("=== TOKENIZATION COMPARISON ===")

  enc = tiktoken.get_encoding("cl100k_base")
  